# Dragapult action-policy prior (Mazlum BC pipeline)

Bu notebook Mazlum’un **option-feature + card embedding + masked cross-entropy** yaklaşımını kullanır; hedef value model değil, **legal option başına logit üreten action prior**.

## Girdi
Kaggle’a `mazlum-dragapult-v127` dataset’ini ekle (içinde `dataset_dragapult_v127.npz` ve `code/src/mazlum/...` var).

## Çıktı
- `/kaggle/working/policy_prior.npz`
- model boyutu/val top-1/metrics

Bu model daha sonra lean4 search’ünde **candidate prior** olarak kullanılacak; kural policy’yi tek başına ezmeyecek. Arena/erken durdurma kapalı; bu notebook sadece policy prior üretir.


In [ ]:
from pathlib import Path
import subprocess, sys

INPUT=Path('/kaggle/input/mazlum-dragapult-v127')
if not INPUT.exists():
    hits=list(Path('/kaggle/input').rglob('dataset_dragapult*.npz'))
    if not hits: raise RuntimeError('dataset_dragapult*.npz bulunamadı; mazlum-dragapult-v127 datasetini ekle.')
    INPUT=hits[0].parent

DATA=INPUT/'dataset_dragapult_v127.npz'
if not DATA.exists():
    DATA=sorted(INPUT.rglob('dataset_dragapult*.npz'))[0]
CODE=INPUT/'code'
TRAIN=CODE/'src'/'mazlum'/'train_bc.py'
if not TRAIN.exists():
    hits=list(Path('/kaggle/input').rglob('train_bc.py'))
    if not hits: raise RuntimeError('train_bc.py bulunamadı; code/ datasetini ekle.')
    TRAIN=hits[0]
print('DATA',DATA)
print('TRAIN',TRAIN)
print('dataset MB',round(DATA.stat().st_size/1e6,1))

## Dataset kontrolü

In [ ]:
import numpy as np
d=np.load(DATA,mmap_mode='r')
print({k:(d[k].shape,str(d[k].dtype)) for k in d.files})
print('rows',d['state'].shape[0],'episodes',np.unique(d['episode_id']).size if 'episode_id' in d.files else '?')

## Policy prior eğitimi

Varsayılan: 12 epoch, batch 64, lr 0.001, arena kapalı. Modelin 1.14M parametreli olması durumunda GPU açık olmalı.

In [ ]:
OUT=Path('/kaggle/working/policy_prior.npz')
cmd=[sys.executable,str(TRAIN),'--dataset',str(DATA),'--out',str(OUT),'--epochs','12','--batch-size','64','--lr','0.001','--arena-every','0','--print-every','1']
print(' '.join(cmd)); subprocess.run(cmd,check=True)

In [ ]:
from IPython.display import FileLink,display
print('exists',OUT.exists(),'MB',round(OUT.stat().st_size/1e6,2) if OUT.exists() else None)
if OUT.exists():
    with np.load(OUT) as z: print('arrays',len(z.files),'params approx',sum(z[k].size for k in z.files))
    display(FileLink(str(OUT)))


## Not

Bu çıktı `policy_prior.npz`; doğrudan submission değil. Bir sonraki adımda lean4 search’ünde legal option skorlarına küçük bir prior olarak bağlanacak. Lethal override ve kural policy fallback korunacak.
